In [2]:
import pandas as pd

df = pd.read_csv("data/raw_race_results.csv")
print(df.shape)
df.head()

(3458, 13)


,Season,Round,EventName,Country,DriverId,Driver,FullName,Team,GridPosition,FinishPosition,ClassifiedPosition,Status,Points
0,2018,1,Australian Grand Prix,Australia,vettel,VET,Sebastian Vettel,Ferrari,3.0,1.0,1,Finished,25.0
1,2018,1,Australian Grand Prix,Australia,hamilton,HAM,Lewis Hamilton,Mercedes,1.0,2.0,2,Finished,18.0
2,2018,1,Australian Grand Prix,Australia,raikkonen,RAI,Kimi Räikkönen,Ferrari,2.0,3.0,3,Finished,15.0
3,2018,1,Australian Grand Prix,Australia,ricciardo,RIC,Daniel Ricciardo,Red Bull Racing,8.0,4.0,4,Finished,12.0
4,2018,1,Australian Grand Prix,Australia,alonso,ALO,Fernando Alonso,McLaren,10.0,5.0,5,Finished,10.0


In [3]:
df["FinishPosition"] = pd.to_numeric(df["FinishPosition"], errors="coerce")
# turns finishposition col into numeric values and anything that isnt a number will be labeled as NaN (Not a number)
df["Podium"] = (df["FinishPosition"] <= 3).astype(int) 
# true becomes 1, so any positions 1,2,3
# false becomes 0, anything lower
podiums_per_race = df.groupby(["Season", "Round"])["Podium"].sum()
print(podiums_per_race.value_counts())
#grouping by race and summing the Podium column should give you 3 for almost every single race (since exactly 3 drivers podium), 
#so if value_counts() shows a lot of 2s or 1s, something's off in the underlying data

Podium
3    173
Name: count, dtype: int64


In [4]:
# from this output we can see that every race has 3 finishers and there are roughly 149 races from 
# now we have to feature engineer the data set to give the model something useful to learn from 
# what comes before the grand prix itself is the grid position which usually can give a good idea of what we can expect the grand prix podium to look like

df = df.sort_values(["DriverId", "Season", "Round"]).reset_index(drop=True)
# building a rolling average for each driver
df["DriverFormLast5"] = ( 
    # avg finishing position over last 5 races for a driver, its a new col we are creating
    df.groupby("DriverId")["FinishPosition"]
    .transform(lambda x: x.shift(1).rolling(window=5, min_periods=1).mean())
)

In [5]:
df[["DriverId", "Season", "Round", "FinishPosition", "DriverFormLast5"]].head(10)

# in the output, ppl with NaN have no race history because its their first race yet

,DriverId,Season,Round,FinishPosition,DriverFormLast5
0,aitken,2020,16,16.0,NaN
1,albon,2019,1,14.0,NaN
2,albon,2019,2,9.0,14.0
3,albon,2019,3,10.0,11.5
4,albon,2019,4,11.0,11.0
5,albon,2019,5,11.0,11.0
6,albon,2019,6,8.0,11.0
7,albon,2019,7,19.0,9.8
8,albon,2019,8,15.0,11.8
9,albon,2019,9,15.0,12.8


In [6]:
df = df.sort_values(["Team", "Season", "Round"]).reset_index(drop=True)
# building a rolling average for each team
df["TeamFormLast5"] = ( 
    # avg finishing position over last 5 races for a team, its a new col we are creating
    df.groupby("Team")["FinishPosition"]
    .transform(lambda x: x.shift(1).rolling(window=5, min_periods=1).mean())
)
#test team
df[["Team", "Season", "Round", "FinishPosition", "TeamFormLast5"]].head()


,Team,Season,Round,FinishPosition,TeamFormLast5
0,Alfa Romeo,2022,1,6.0,NaN
1,Alfa Romeo,2022,1,10.0,6.000000
2,Alfa Romeo,2022,2,15.0,8.000000
3,Alfa Romeo,2022,2,11.0,10.333333
4,Alfa Romeo,2022,3,8.0,10.500000


In [7]:
df = df.sort_values(["DriverId", "Country", "EventName"]).reset_index(drop=True)
df["TrackHistory"] = (
    df.groupby("DriverId")["FinishPosition"]
    .transform(lambda x: x.shift(1).rolling(window=5, min_periods=1).mean())
)
df[["DriverId", "Season", "Round", "FinishPosition", "TrackHistory"]].head()

,DriverId,Season,Round,FinishPosition,TrackHistory
0,aitken,2020,16,16.0,NaN
1,albon,2019,21,6.0,NaN
2,albon,2020,17,4.0,6.000000
3,albon,2022,22,13.0,5.000000
4,albon,2023,22,14.0,7.666667


In [8]:
# adding more features, now that we have another dataset

In [9]:
# reading the qualifying dataset
quali_df = pd.read_csv("data/qualifying_data.csv")

# merging the qualifying data in
df = df.merge(
    quali_df[["Season", "Round", "DriverId", "QualiGapToPole"]],
    on=["Season", "Round", "DriverId"],
    how="left"
)

print("QualiGapToPole NaNs:", df["QualiGapToPole"].isna().sum())
print(df["QualiGapToPole"].describe())

QualiGapToPole NaNs: 45
count    3413.000000
mean        1.562425
std         2.727584
min       -17.098000
25%         0.604000
50%         1.269000
75%         2.084000
max        62.297000
Name: QualiGapToPole, dtype: float64


In [11]:
# Mapping of DriverId to their nationality (country code)
# FastF1 uses these abbreviations in the CountryCode field
home_race_map = {
    "VER": "Netherlands", "HAM": "United Kingdom", "LEC": "Monaco",
    "SAI": "Spain", "RUS": "United Kingdom", "NOR": "United Kingdom",
    "PIA": "Australia", "ALO": "Spain", "STR": "Canada",
    "GAS": "France", "OCO": "France", "ALB": "Thailand",
    "BOT": "Finland", "RIC": "Australia", "TSU": "Japan",
    "MAG": "Denmark", "HUL": "Germany", "ZHO": "China",
    "LAW": "New Zealand", "COL": "Argentina", "ANT": "Brazil",
    "BEA": "United Kingdom", "DOO": "Australia", "HAD": "United States",
    # add others as needed based on your DriverId values in the data
}

df["HomeRace"] = (
    df.apply(lambda row: home_race_map.get(row["Driver"], "") == row["Country"], axis=1)
).astype(int)

print("Home race rows:", df["HomeRace"].sum())

Home race rows: 66


In [12]:
# this counts how many of a driver's last 5 races they finished in the points/top 10
# so, value of 5 means they've scored points in every recent race


df["InPointsLast5"] = df["FinishPosition"].apply(lambda x: 1 if x <= 10 else 0)

df = df.sort_values(["DriverId", "Season", "Round"]).reset_index(drop=True)

df["PointsFinishesLast5"] = (
    df.groupby("DriverId")["InPointsLast5"]
    .transform(lambda x: x.shift(1).rolling(window=5, min_periods=1).sum())
)

df.drop(columns=["InPointsLast5"], inplace=True)
print(df[["Driver", "Season", "Round", "PointsFinishesLast5"]].head(10))

  Driver  Season  Round  PointsFinishesLast5
0    AIT    2020     16                  NaN
1    ALB    2019      1                  NaN
2    ALB    2019      2                  0.0
3    ALB    2019      3                  1.0
4    ALB    2019      4                  2.0
5    ALB    2019      5                  2.0
6    ALB    2019      6                  2.0
7    ALB    2019      7                  3.0
8    ALB    2019      8                  2.0
9    ALB    2019      9                  1.0


In [16]:
# Points accumulated before each race (shifted so no leakage)
df = df.sort_values(["Season", "Round", "DriverId"]).reset_index(drop=True)

df["CumulativeDriverPoints"] = (
    df.groupby(["Season", "DriverId"])["Points"]
    .transform(lambda x: x.shift(1).cumsum().fillna(0))
)

df["CumulativeTeamPoints"] = (
    df.groupby(["Season", "Team"])["Points"]
    .transform(lambda x: x.shift(1).cumsum().fillna(0))
)

print(df[["Driver", "Season", "Round", "CumulativeDriverPoints", "CumulativeTeamPoints"]].head(10))

  Driver  Season  Round  CumulativeDriverPoints  CumulativeTeamPoints
0    ALO    2018      1                     0.0                   0.0
1    BOT    2018      1                     0.0                   0.0
2    HAR    2018      1                     0.0                   0.0
3    ERI    2018      1                     0.0                   0.0
4    GAS    2018      1                     0.0                   0.0
5    GRO    2018      1                     0.0                   0.0
6    HAM    2018      1                     0.0                   4.0
7    HUL    2018      1                     0.0                   0.0
8    MAG    2018      1                     0.0                   0.0
9    LEC    2018      1                     0.0                   0.0


In [24]:
fill_value = df["FinishPosition"].median()
print("Filling NaNs with median finish position:", fill_value)

for col in ["DriverFormLast5", "TeamFormLast5", "TrackHistory", "GridPosition", "FinishPosition"]:
    nan_count = df[col].isna().sum()
    df[col] = df[col].fillna(fill_value)
    print(f"  {col}: filled {nan_count} NaN rows")

Filling NaNs with median finish position: 10.0
  DriverFormLast5: filled 0 NaN rows
  TeamFormLast5: filled 0 NaN rows
  TrackHistory: filled 0 NaN rows
  GridPosition: filled 0 NaN rows
  FinishPosition: filled 3 NaN rows


In [25]:
# Fill NaNs — QualiGapToPole gets its own fill since scale is different
df["QualiGapToPole"] = df["QualiGapToPole"].fillna(df["QualiGapToPole"].median())

fill_value = df["FinishPosition"].median()
for col in ["DriverFormLast5", "TeamFormLast5", "TrackHistory", "PointsFinishesLast5"]:
    df[col] = df[col].fillna(fill_value)

# Final feature table
feature_cols = [
    "GridPosition",
    "QualiGapToPole",
    "DriverFormLast5",
    "TeamFormLast5",
    "TrackHistory",
    "PointsFinishesLast5",
    "HomeRace",
    "CumulativeDriverPoints",
    "CumulativeTeamPoints",
]
id_cols = ["Season", "Round", "DriverId", "Driver", "Country"]
label_col = "Podium"

final_df = df[id_cols + feature_cols + [label_col]].copy()
final_df.to_csv("data/features.csv", index=False)
print(final_df.shape)
final_df.head()

(3458, 15)


,Season,Round,DriverId,Driver,Country,GridPosition,QualiGapToPole,DriverFormLast5,TeamFormLast5,TrackHistory,PointsFinishesLast5,HomeRace,CumulativeDriverPoints,CumulativeTeamPoints,Podium
0,2018,1,alonso,ALO,Australia,10.0,2.528,10.0,10.0,10.4,10.0,0,0.0,0.0,0
1,2018,1,bottas,BOT,Australia,15.0,0.925,10.0,10.0,8.2,10.0,0,0.0,0.0,0
2,2018,1,brendon_hartley,HAR,Australia,16.0,3.368,10.0,10.0,10.0,10.0,0,0.0,0.0,0
3,2018,1,ericsson,ERI,Australia,17.0,3.392,10.0,10.0,10.0,10.0,0,0.0,0.0,0
4,2018,1,gasly,GAS,Australia,20.0,4.131,10.0,15.0,11.4,10.0,0,0.0,0.0,0


In [26]:
print(df.isnull().sum())


Season                    0
Round                     0
EventName                 0
Country                   0
DriverId                  0
Driver                    0
FullName                  0
Team                      0
GridPosition              0
FinishPosition            0
ClassifiedPosition        0
Status                    0
Points                    0
Podium                    0
DriverFormLast5           0
TeamFormLast5             0
TrackHistory              0
QualiGapToPole            0
HomeRace                  0
PointsFinishesLast5       0
CumulativeDriverPoints    0
CumulativeTeamPoints      0
dtype: int64
